<a href="http://landlab.github.io"><img style="float: left" src="https://raw.githubusercontent.com/landlab/tutorials/release/landlab_header.png"></a>

# <span style="color: green;">Your First Landlab Model</span>

In this notebook, you'll create a landscape and then run the `LinearDiffuser` component on that grid.

# Generating specific topography: a mound on a hillslope

Now that we know how to make a GridObject, let's generate a more complex initial GridObject and run the Landlab Component [`LinearDiffuser`](https://landlab.csdms.io/generated/api/landlab.components.diffusion.diffusion.html#landlab.components.diffusion.diffusion.LinearDiffuser) on it.

### First, let's create some functions to help us create a mound and plot. 
Don't worry about trying to understand these for now, just run the cell. We'll have time to dive into plotting things later.


In [ ]:
from matplotlib import cm
import matplotlib.colors as mcolors

def create_mounds(grid, mound_centers, mound_height, mound_radius, taper_coefficient):
    """
    Add one or more Gaussian-shaped mounds to a grid's topographic elevation.

    Each mound is centered on a given node and adds elevation that peaks at the center and tapers smoothly to 
    zero with distance, based on a Gaussian falloff.

    Parameters
    ----------
    grid : RasterModelGrid
        The Landlab grid to modify. Must already have a 'topographic__elevation' field at nodes.
    mound_centers : int or array_like of int
        Node ID(s) marking the center of each mound.
    mound_height : float
        Maximum elevation added at the center of each mound, in the grid's
        elevation units (e.g. meters).
    mound_radius : float
        Distance from each mound's center beyond which no elevation is added, in the grid's spatial units 
        (e.g. meters).
    taper_coefficient : float
        Controls how quickly the mound tapers off within mound_radius. 
        Larger values produce a more gradual, spread-out taper; smaller values produce a sharper, more peaked mound.

    Returns
    -------
    None
        Modifies the grid's 'topographic__elevation' field in place.

    Notes
    -----
    Mounds are added on top of whatever elevation already exists at each node, so calling this function 
    multiple times, or passing multiple mound_centers, will stack mounds rather than overwrite the surface.
    """
    # add a mounds
    raised_nodes = np.array([mound_centers])       # node IDs to raise - this is the center of the mound; can change; can make it alist
    sigma = mound_radius / taper_coefficient        # controls how quickly the mound tapers off (Gaussian spread)
    
    # add a Gaussian bump around each raised node (set upt so that you can have many mound)
    for node in raised_nodes:
        xc, yc = grid.x_of_node[node], grid.y_of_node[node]                        # coordinates of this mound's center
        distances = np.sqrt((grid.x_of_node - xc)**2 + (grid.y_of_node-yc)**2)     # distance from every node in the grid to the mound center
        mask = distances < mound_radius                                            # only affect the nodes within the bump_radius of the center
        mound = mound_height * np.exp(-((distances[mask])**2) / (2* sigma**2))     # Gaussian falloff: highest at the center, tapering to 0 at the edges
        grid.at_node['topographic__elevation'][mask] += mound                      # add the bump on top of the existing elevation


def surf_plot(grid, surface="topographic__elevation", units='m', title="Topographic Elevation (3D)", elev=20, azim=-150, ax=None, cmap='gray', vmin=None, vmax=None):
    """
    Plot a Landlab grid's surface as a shaded 3D topography plot.

    Parameters
    ----------
    grid : RasterModelGrid
        The Landlab grid containing the surface to plot.
    surface : str, optional
        Name of the grid field to plot (default is "topographic__elevation").
    units : str, optional
        Units label to display on the axes (default is 'm').
    title : str, optional
        Title displayed above the plot (default is "Topographic Elevation (3D)").
    azim : float, optional
        Azimuthal viewing angle in degrees, controlling the plot's rotation
        (default is -150).
    ax : matplotlib.axes.Axes3D, optional
        An existing 3D axes to plot into. If None (default), a new standalone figure and axes are created.
    cmap : str, optional
        Name of the Matplotlib colormap used to color the surface by elevation
        (default is 'gray').
    vmin, vmax : float, optional
        Fixed lower/upper bounds for the z-axis and colorbar. If None (default), bounds are set automatically from the data's own
        min/max. Useful for anchoring the axes and colorbar to the same range across multiple calls (e.g. comparing several time steps),
        so scale doesn't shift from plot to plot.

    Returns
    -------
    None
        Displays the plot. If `ax` is provided, the plot is drawn into that
        axes and the figure is not shown automatically (the caller is
        responsible for calling plt.show()).

    Notes
    -----
    The colorbar range is explicitly normalized to the true min/max of the
    surface data using `norm`. Without this, plot_surface colors each face
    by averaging its corner values, which can visually compress the color
    range compared to a 2D plot of the same data (e.g. imshow_grid).
    """
    single_fig = ax is None
    if single_fig:
        fig = plt.figure(figsize=(10, 6))
        ax = fig.add_axes([0.05, 0.05, 0.75, 0.90], projection="3d") # for some reason this plot cuts off the z-axis label for certain 
                                                                      # orientations (azim=-70) so this forces it to a specific position for some of them 
    else:
        fig = ax.get_figure()
    # plot the surface
    z = grid.at_node[surface].reshape(grid.shape)

    # Use provided vmin/vmax if given, otherwise fall back to this data's own min/max
    if vmin is not None:
        zmin = vmin
    else:
        zmin = z.min()
    if vmax is not None:
        zmax = vmax
    else:
        zmax = z.max()

    norm = mcolors.Normalize(vmin=zmin, vmax=zmax)
    surf = ax.plot_surface(
                grid.x_of_node.reshape(grid.shape),
                grid.y_of_node.reshape(grid.shape),
                z,
                cmap=cmap,
                norm=norm,
                linewidth=0.0,
                antialiased=False,
    )
    ax.view_init(elev=elev, azim=azim)
    ax.set_zlim(zmin, zmax)
    ax.set_xlabel(f"x [{units}]")
    ax.set_ylabel(f"y [{units}]")
    ax.set_zlabel(f"elevation [{units}]")
    plt.title(title)
    fig.colorbar(surf, ax=ax, label=f"elevation [{units}]")
    if single_fig:
        plt.show()

def compare_2d_3d(grid, surface="topographic__elevation", units='m', elev=20, azim=-70 , cmap_3d='gray', vmin=None, vmax=None, plot_type=None):
    """
    Plot a Landlab grid's surface side by side as a 2D map view and a 3D
    shaded topography plot.

    Parameters
    ----------
    grid : RasterModelGrid
        The Landlab grid containing the surface to plot.
    surface : str, optional
        Name of the grid field to plot (default is "topographic__elevation").
    units : str, optional
        Units label to display on the 3D plot's axes (default is 'm').
    elev : float, optional
        Elevation viewing angle in degrees for the 3D plot, controlling
        how high the camera sits above the surface (default is 20).
    azim : float, optional
        Azimuthal viewing angle in degrees for the 3D plot, controlling its rotation (default is -70).
    cmap_3d : str, optional
        Name of the Matplotlib colormap used to color the 3D surface by elevation (default is 'gray').
    vmin, vmax : float, optional
        Fixed lower/upper bounds for the 3D plot's z-axis and colorbar. If None (default), bounds are set automatically from this surface's
        own min/max. Useful for anchoring the scale across multiple calls (e.g. comparing several time steps) so it doesn't shift between
        plots.
    plot_type : str, optional
        If set to 'hillshade', the 2D panel is drawn using Landlab's `imshowhs_grid` (shaded relief) instead of the default flat-color
        `imshow_grid`. Any other value (including None, the default) uses `imshow_grid`.

    Returns
    -------
    None
        Displays a single figure containing both the 2D and 3D plots side by side.

    Notes
    -----
    The 2D panel uses Landlab's `imshow_grid` (or `imshowhs_grid` if plot_type='hillshade'), which draws to whichever axes is "current" —
    `plt.sca(ax1)` is used to point it at the left subplot before calling it. The 3D panel is drawn using `surf_plot`, passed the right subplot
    directly via its `ax` argument.
    fig = plt.figure(figsize=(10, 4))

    # left panel: 2D imshow
    ax1 = fig.add_subplot(1, 2, 1)
    plt.sca(ax1)  # tells imshow_grid to draw here
    if plot_type == 'hillshade':
        imshowhs_grid(grid, "topographic__elevation")
    else:
        imshow_grid(grid, surface, colorbar_label="Elevation (m)")
    ax1.set_title("Topographic Elevation (2D)")

    # right panel: 3D surface
    ax2 = fig.add_subplot(1, 2, 2, projection="3d")
    surf_plot(grid, surface=surface, units=units, azim=azim, ax=ax2, cmap=cmap_3d, vmin=vmin, vmax=vmax)

    plt.tight_layout()
    plt.show()

### Now let's make a flat surface with a mound in the middle of it.
The cell below will generate a gentle hilslope with a towering mound in the middle of it. (We know this isn't realistic, but bear with us.)

In [ ]:
# --------------Importing Libraries----------------------------
import numpy as np
import matplotlib.pyplot as plt
from landlab import RasterModelGrid, imshow_grid
from landlab.components import LinearDiffuser

# --------------Setting Up a Grid----------------------------
# make a GridObject
grid = RasterModelGrid((25, 25), xy_spacing=10.0)

# make a field and initialize it
topo = grid.add_zeros('topographic__elevation', at='node')

# create a spikey mound at the center of our grid
create_mounds(grid, mound_centers=[312], mound_height=50, mound_radius=50, taper_coefficient=5)

# plotting
compare_2d_3d(grid, azim=-70,cmap_3d='terrain', vmin=0, vmax=50)


### Time to model diffusion!

The cell below is initially set up to simulate 10 kyr with 10 year timesteps and will plot every 1000 years. *Note: This cell assumes that you've created the grid above.*

In [ ]:
# --------------Model----------------------------
# instantiate component
ld = LinearDiffuser(grid, linear_diffusivity=0.02)

# set up time-stepping
time_step     = 10      # years per model step
total_time    = 10000   # total years to simulate
plot_interval = 1000    # how often (in years) to print progress and plot

# run model
for i in range(0, total_time+1, time_step):
    ld.run_one_step(time_step) # runs the LinearDiffuser which will simulate soil creep and hillslope processes

    # plot at the plot_interval to show the current state of hte hillslope
    if i % plot_interval == 0:
        print(f"Time: {i} yr")
        compare_2d_3d(grid, azim=-70, cmap_3d='terrain', vmin=0, vmax=50)

## How would we export this "end result" grid?
Here's one way:
```python
import pickle # import pickle

with open('folder/grid.pkl', 'wb') as f: # save your grid
    pickle.dump(grid, f)                 # this setup will automatically save all fields
```


<div class="alert alert-success">

## **Tips** :

|question|what to do|python |
|--|--|--|
|Want to quickly get **documention** for something in a Jupyter Notebook?|Set your cursor on what you want info for and then press `Shift` + `Tab` and the documentation should appear.|
|Want to **comment out a lot of lines** in a Jupyter Notebook?|Highlight the lines you want commented out and then press `ctrl` + `/` ||
|Want to **set your units** within your GridObject?|When you first create your GridObject using `RasterModelGrid`, add `xy_axis_units`. You can use whatever units you want, just be internally consistent. |<pre style="background:#f5f0e8;padding:8px;border-radius:4px;margin:0;"><code>grid = RasterModelGrid((40, 60), xy_spacing=100.0, xy_axis_units='m')</code></pre>
|Want to generate the **same random pattern** as your friends?|`np.random.rand()` will generate a **new** random pattern every time the cell is run. To recreate the **same random pattern each time** add this line anywhere before the use of this numpy method. You can change the value of seed to generate different random patterns.|<pre style="background:#f5f0e8;padding:8px;border-radius:4px;margin:0;"><code>np.random.seed(seed=5000)</code></pre>|
|Want to **change the size of your figure**?|Make sure you've imported `matplotlib.pyplot` and then just add this to define your figure size before you use `imshow_grid`. Like other uses of `figsize`, you can change those to be whatever dimensions you want.|<pre style="background:#f5f0e8;padding:8px;border-radius:4px;margin:0;"><code>plt.figure(figsize=(10,5))</code></pre>|
|Does having the **colorbar size** be larger than your figure annoy you?|In your use of `imshow_grid()` add `shrink`. The value will be the fraction by which to shrink the colorbar. |<pre style="background:#f5f0e8;padding:8px;border-radius:4px;margin:0;"><code>imshow_grid(grid, topo, colorbar_label="Elevation (m)", shrink=.7)</code></pre>
|Want to **access something in a different folder** than where you are?|Use the `..` prefix to go up a folder level |<pre style="background:#f5f0e8;padding:8px;border-radius:4px;margin:0;"><code>with open('../folder/grid.pkl', 'wb')</code></pre>

----------
# What about real data?
Here, we will use the bmi-topography package to import a DEM. This package and Python library will be used to access and import the NASA Shuttle Radar Topography Mission (SRTM) land elevation data, which we will use to run the `LinearDiffuser` over.

<span style="font-size:0.85em;color:#6b7a8d;"><em>For more information on this package and how to use it yourself, follow these links: [bmi-topography repo](https://github.com/csdms/bmi-topography) and [notebook on bmi-topography](https://github.com/csdms/bmi-topography/blob/main/examples/topography.ipynb).</em></span>


### Making sure we have all the libraries we need imported...

In [ ]:
from matplotlib.colors import SymLogNorm           # will help us make the plot scales in a symmetrical log scale
from bmi_topography import Topography              # how we will get the DEM for the exercise
from landlab.io import read_esri_ascii             # will be used to read the DEM into a RasterModelGrid
from landlab import imshowhs_grid                  # how we will display the various outputs from the model

### We will then define the variable topo, which will create an instance of topography, and we will define the following parameters:

1. the type of data we are requesting - in this case SRTMGL1 (SRTM Global Dataset 1, resolution ~30 m)
2. the geographic bounding (south, north, west, east) of the data
3. the file format we want - in this case we want as an ascii file
4. where to store the file

In [ ]:
#-----defining the DEM that we'll grab from OpenTopography--------
topo = Topography(
    dem_type="SRTMGL1",
    south=39.93,
    north=40.0,
    west=-105.33,
    east=-105.26,
    output_format="AAIGrid",
    cache_dir="DEMData//",
)
fname = topo.fetch() # downloads data
dem = topo.load()    # commits data to memory

#-----Looking at the the DEM-------
grid_geog, elev = read_esri_ascii(fname, name="topographic__elevation") # reading in the data, calling the grid object grid_geog and assigning topo to elev variable

grid_geog.imshow( # visualizing the topo elevation using imshow!
    "topographic__elevation", 
    cmap="terrain",
    grid_units=("deg", "deg"),
    colorbar_label="Elevation (m)",
)

<div class="alert alert-success">

## **Note**:
Simply creating the `topo` variable and create an instance of topography does not download the data. We will also need to use `fetch` to download the data and `load` to commit the data to memory

## But it's not Landlab compatible yet...

Looking at this visualization, you will notice that the X and Y axes are in degrees. This is telling us that the DEM is currently in a geographic coordinate system. We actuallyl want the DEM, and soon to be raster model grid to be in a projected coordinate system for what we want to do.

The cell below will do all of this in one step, converting the DEM into a projected coordinate system *AND* reading it in as a Landlab GridObject (a `RasterModelGrid`) with spacing of 30 m (the resolution of the SRTM data). We will also define the topographic elevation to be recorded at the nodes.

In [ ]:
# make a new grid with RasterModelGrid, use the dimensions of grid_geog
grid2 = RasterModelGrid(grid_geog.shape, xy_spacing=30.0)
grid2.at_node["topographic__elevation"] = grid_geog.at_node["topographic__elevation"]

imshowhs_grid(grid2, "topographic__elevation"); # let's visualize this new LandLab grid object! 

## Now let's run the `LinearDiffuser` over this landscape!

In [ ]:
# --------------Model----------------------------
# instantiate component
ld = LinearDiffuser(grid2, linear_diffusivity=0.05)

# set up time-stepping
time_step     = 10 # years per model step
total_time    = 500000 # total years to simulate
plot_interval = 100000 # how often (in years) to print progress and plot

# run model
for i in range(0, total_time+1, time_step):
    ld.run_one_step(time_step) # runs the LinearDiffuser which will simulate soil creep and hillslope processes

    # plot at the plot_interval to show the current state of hte hillslope
    if i % plot_interval == 0:
        print(f"Time: {i} yr")
        compare_2d_3d(grid2, elev=50, azim=-300, cmap_3d='terrain', plot_type='hillshade')

-----
# BONUS: Loading a DEM (not from Open Topography)
Have a DEM file you just want to load? Here are some ways to import different file types.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from landlab import RasterModelGrid, imshowhs_grid
from landlab.io import read_esri_ascii

os.getcwd()

### Importing an ESRI ASCII Grid format (`.asc`) using [read_esri_ascii()](https://landlab.readthedocs.io/en/latest/generated/api/landlab.io.esri_ascii.html)
Earlier, we used `read_esri_ascii` from landlab.io to read in an Open Topography file. You can use the same function to read in any `.asc.` file. 
- note to susannah/etha/mark: we don't have an ASCII file in the ExploreHub right now, but there is one in 'ivy/ESPIn_repos/ESPIn_Floods_2025/DEMData/SRTMGL1_39.93_-105.33_40.0_-105.26.asc' but even that is actually from Open Topography and just saved during a previous groups' project

In [ ]:
grid3, elevs = read_esri_ascii('file_name.asc', name='topographic__elevation')

imshowhs_grid(grid3, "topographic__elevation")

### Importing a TIFF using [Rasterio](https://rasterio.readthedocs.io/en/stable/)

In [ ]:
with rasterio.open("../../../../data/boulder39p5N-106W_NASADEM.tif") as src:
    dem = src.read(1) # reads the first band as a 2D array
    dx = src.res[0]   # pixel width in the fiel's native units (often degrees for NASADEM)
    dy = src.res[1]   # pixel height

# create a grid that is the same shape as the dem with the same spacing
grid4 = RasterModelGrid(dem.shape, xy_spacing=(dx, dy))

# grab the dem data and add it to the `topographic__elevation` field
grid4.add_field('topographic__elevation', dem, at='node')

# visualize it
#imshowhs_grid(grid4, "topographic__elevation")

#### Remember there might be an issue with projections!
NASADEM data is typically in geographic coordinates (lat/lon degrees), not meters. `src.res` will give you spacing in *degrees*, not meters. If you build the grid with that spacing directly, your x & y axes will be in degrees and any distance-based calculations will be wrong...you'll have to reproject.

**Pixel size** = real-world distance represented by one pixel

For this DEM:

- At this latitude (40° N):
    - 1° longitude   ≈ 85 km (varies by latitude)
    - 1° of latitude ≈ 111.2 km (constant everywhere)
- Array size: 401 columns, 315 rows

**Calculating pixel size:**

In [ ]:
# NASADEM uses 1 arcsecond (1/3600 degree) pixels
pixel_size_deg = 1.0 / 3600

# From GeoTIFF metadata: ~40.5°N, -106°W
lat_center = 40.5

# Calculate km per degree at this latitude
lat_km_per_degree = 111.320  # constant
lon_km_per_degree = 111.320 * np.cos(np.radians(lat_center))

# Convert to meters
dx2 = pixel_size_deg * lon_km_per_degree * 1000  # ~23.5m
dy2 = pixel_size_deg * lat_km_per_degree * 1000  # ~30.9m

print(f"Pixel length (E-W): {dx2:.1f} meters")
print(f"Pixel length (N-S): {dy2:.1f} meters")

**Note:** Pixels for this DEM are NOT square! They're rectangular! This is often the case on real-world DEMs due to lat/long coordinates.

In [ ]:
# create a grid that is the same shape as the dem with the same spacing
grid5 = RasterModelGrid(dem.shape, xy_spacing=(dx2, dy2))

# grab the dem data and add it to the `topographic__elevation` field
grid5.add_field('topographic__elevation', dem, at='node')

# visualize it
imshowhs_grid(grid5, "topographic__elevation")